# Automatic Encoding for the Iceberg Code 

We have seen in the Steane code example that Guppy FT provides automatic encoding for Guppy programs. In this example, we will demonstrate automatic encoding for the $[[k+2, k, 2]]$ Iceberg code. Unlike Steane, encoding programs with the Iceberg code requires the use of Nexus cloud.

As Nexus cloud is required, we will need the `qnexus` client package.

```shell
uv add qnexus
```

In addition, we will also need a Quantinuum Nexus account. 


Once we have the `qnexus` package installed, we can login to Nexus and initialize a project to keep track of all of the jobs we run.

In [ ]:
import qnexus as qnx

qnx.login()

project = qnx.projects.get_or_create(name="iceberg_project")
qnx.context.set_active_project(project)

We can now define our computational program using Guppy. Here we define a `prepare_ghz_state` function which prepares a GHZ state over $n$ qubits.

In [ ]:

from guppylang import guppy
from guppylang.std.quantum import qubit, h, cx, measure_array, collect_measurements
from guppylang.std.platform import output
from guppylang.defs import GuppyFunctionDefinition



def build_ghz_main(n_qubits: int) -> GuppyFunctionDefinition:
    @guppy
    def prepare_ghz_state() -> array[qubit, n_qubits]:
        qs = array(qubit() for _ in range(n_qubits))
        h(qs[0])
        for i in range(n_qubits - 1):
            cx(qs[i], qs[i + 1])
        return qs

    @guppy
    def main() -> None:
        ghz_qubits: array[qubit, 16] = prepare_ghz_state(16)
        measurements = measure_array(ghz_qubits)
        output("m", collect_measurements(measurements))

    return main

In [12]:
from guppyft.code.iceberg.encode import IcebergBuilder

pkg = main.compile()
IcebergBuilder().build(block_size=2, n_blocks=8).annotate(pkg)

Error: Array comprehension with nonstatic size (at <In[11]>:7:14)
  | 
5 | @guppy
6 | def prepare_ghz_state[n: nat](n_qubits: int @comptime) -> array[qubit, n]:
7 |     qs = array(qubit() for _ in range(n_qubits))
  |               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^ Cannot infer the size of this array comprehension ...

Note:
  | 
6 | def prepare_ghz_state[n: nat](n_qubits: int @comptime) -> array[qubit, n]:
7 |     qs = array(qubit() for _ in range(n_qubits))
  |                                 --------------- since the number of elements yielded by this iterator is not
  |                                                 statically known

Guppy compilation failed due to 1 previous error
